# 07 — Multi-site UAV Musanga fractional cover × Google Satellite Embeddings

## Question
Can annual Google Satellite Embeddings predict **UAV-derived fractional Musanga cover** across multiple landscapes?

## Design
- UAV binary masks are the reference truth.
- **Yoko UAV year = 2020**; other products = **2024**.
- Compare **T0** (observation year), **T+1**, and **T0+T1** embeddings.
- Start at **10 m**, then test **20 / 30 / 50 m**.
- Primary continuous metrics: RMSE, MAE, R², Spearman ρ, bias.
- Detection diagnostics at observed cover >1%, >5%, >20%, >50%: ROC-AUC, PR-AUC; F1/precision/recall/balanced accuracy use a training-selected operating threshold.
- Preserve zeros; require high valid-UAV support.
- Cache expensive target and embedding products to Drive.
- Sentinel-2 is deliberately deferred until the embedding-only baseline is understood.

The Mos mosaics are kept as separate products, while evaluation code also carries a landscape ID so adjacent/overlapping Mos products can be grouped for strict transfer tests.

## BLOCK 01 — Setup, Drive and configuration

In [ ]:
!pip -q install earthengine-api geemap rasterio pyproj scikit-learn scipy joblib

from pathlib import Path
import re, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, balanced_accuracy_score)
import joblib, ee, geemap

from google.colab import drive
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
if not MYDRIVE.exists() or not any(MYDRIVE.iterdir()):
    drive.mount(str(DRIVE_ROOT), force_remount=False)
else:
    print("Google Drive already mounted.")

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"
UAV_DIR=ROOT/"data"/"multisite_uav"
OUT=ROOT/"work"/"multisite_embedding_fractional_cover"
OUT.mkdir(parents=True,exist_ok=True)
EE_PROJECT="tropsedslu"
SUPPORTS=[10,20,30,50]
MIN_UAV_COVERAGE=0.90
RANDOM_SEED=42
FORCE=False

SITE_META={
    "Yoko":{"obs_year":2020,"landscape":"Yoko"},
    "Mos_1_2":{"obs_year":2024,"landscape":"Mos"},
    "Mos_3_4":{"obs_year":2024,"landscape":"Mos"},
    "Mos_5_6":{"obs_year":2024,"landscape":"Mos"},
    "YB_Carlier":{"obs_year":2024,"landscape":"YB_Carlier"},
}
print("UAV folder:",UAV_DIR)
print("Output:",OUT)

## BLOCK 02 — Discover authoritative binary UAV masks

In [ ]:
masks=sorted(UAV_DIR.glob("*_musanga_binary_mask_threshold_*.tif"))
rows=[]
for p in masks:
    m=re.match(r"(.+)_musanga_binary_mask_threshold_([0-9.]+)\.tif$",p.name,re.I)
    if not m: continue
    site=m.group(1); thr=float(m.group(2))
    meta=SITE_META.get(site,{"obs_year":2024,"landscape":site})
    with rasterio.open(p) as src:
        b=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
        rows.append(dict(site=site,landscape=meta["landscape"],obs_year=meta["obs_year"],
                         threshold=thr,path=str(p),crs=str(src.crs),res_x=abs(src.transform.a),
                         res_y=abs(src.transform.e),width=src.width,height=src.height,
                         lon_min=b[0],lat_min=b[1],lon_max=b[2],lat_max=b[3]))
inventory=pd.DataFrame(rows).sort_values("site").reset_index(drop=True)
assert len(inventory)==5, f"Expected 5 masks, found {len(inventory)}"
display(inventory[["site","landscape","obs_year","threshold","res_x","res_y"]])
inventory.to_csv(OUT/"uav_inventory.csv",index=False)

## BLOCK 03 — Earth Engine and embedding cache helpers

In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

BANDS=[f"A{i:02d}" for i in range(64)]

def embedding_path(site,year,support):
    return OUT/"embeddings"/f"{site}_embedding_{year}_{support}m.tif"

def get_embedding(site,mask_path,year,support):
    out=embedding_path(site,year,support); out.parent.mkdir(parents=True,exist_ok=True)
    if out.exists() and not FORCE:
        return out
    with rasterio.open(mask_path) as src:
        left,bottom,right,top=transform_bounds(src.crs,"EPSG:4326",*src.bounds,densify_pts=21)
    roi=ee.Geometry.Rectangle([left,bottom,right,top],geodesic=False)
    col=(ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
         .filterDate(f"{year}-01-01",f"{year+1}-01-01").filterBounds(roi))
    print(site,year,"images:",col.size().getInfo())
    img=col.mosaic().select(BANDS)
    geemap.ee_export_image(img,filename=str(out),scale=support,region=roi,file_per_band=False)
    if not out.exists(): raise RuntimeError(f"Embedding export failed: {out}")
    return out

print("EE ready.")

## BLOCK 04 — Build UAV fractional cover on an embedding grid

The binary mask is averaged directly from native UAV pixels. A separate valid-data raster is averaged to estimate UAV support per satellite cell; cells below 90% valid UAV coverage are excluded.

In [ ]:
def target_paths(site,year,support):
    d=OUT/"targets"; d.mkdir(parents=True,exist_ok=True)
    return d/f"{site}_fraction_{support}m.tif", d/f"{site}_validcoverage_{support}m.tif"

def build_target(mask_path,emb_path,site,year,support):
    frac_path,cov_path=target_paths(site,year,support)
    if frac_path.exists() and cov_path.exists() and not FORCE:
        return frac_path,cov_path
    with rasterio.open(emb_path) as ref:
        shape=(ref.height,ref.width); dst_crs=ref.crs; dst_transform=ref.transform
        profile=ref.profile.copy()
    frac=np.full(shape,np.nan,dtype="float32")
    cov=np.zeros(shape,dtype="float32")
    with rasterio.open(mask_path) as src:
        nd=src.nodata
        reproject(source=rasterio.band(src,1),destination=frac,
                  src_transform=src.transform,src_crs=src.crs,src_nodata=nd,
                  dst_transform=dst_transform,dst_crs=dst_crs,dst_nodata=np.nan,
                  resampling=Resampling.average,init_dest_nodata=True)
        # Validity mask on disk would be safest for huge rasters; dataset_mask is read
        # at destination-scale to avoid loading native billion-pixel arrays.
        valid_native=src.dataset_mask(out_shape=shape,resampling=Resampling.average).astype("float32")/255.0
        # dataset_mask(out_shape) is aligned to source extent, not ref transform; reproject a
        # lightweight source mask via WarpedVRT to the exact ref grid.
        from rasterio.vrt import WarpedVRT
        with WarpedVRT(src,crs=dst_crs,transform=dst_transform,width=shape[1],height=shape[0],
                       resampling=Resampling.average) as vrt:
            cov=vrt.dataset_mask().astype("float32")/255.0
    good=np.isfinite(frac)&(frac>=0)&(frac<=1)&(cov>=MIN_UAV_COVERAGE)
    prof=profile.copy(); prof.update(count=1,dtype="float32",nodata=-9999.0,compress="deflate")
    with rasterio.open(frac_path,"w",**prof) as ds: ds.write(np.where(good,frac,-9999).astype("float32"),1)
    with rasterio.open(cov_path,"w",**prof) as ds: ds.write(np.where(np.isfinite(cov),cov,-9999).astype("float32"),1)
    return frac_path,cov_path

print("Target builder ready.")

## BLOCK 05 — E00/E01: build the canonical 10 m multi-site table for T0 and T+1

In [ ]:
def build_site_table(row,support=10):
    site=row.site; year=int(row.obs_year); mask=Path(row.path)
    e0=get_embedding(site,mask,year,support)
    e1=get_embedding(site,mask,year+1,support)
    frac_path,cov_path=build_target(mask,e0,site,year,support)
    with rasterio.open(frac_path) as ds:
        y=ds.read(1).astype("float32"); nd=ds.nodata; tr=ds.transform
        valid=np.isfinite(y)&(y!=nd)
    with rasterio.open(cov_path) as ds:
        cov=ds.read(1).astype("float32"); cnd=ds.nodata
        valid &= np.isfinite(cov)&(cov!=cnd)&(cov>=MIN_UAV_COVERAGE)
    with rasterio.open(e0) as ds: X0=ds.read().astype("float32"); nd0=ds.nodata
    with rasterio.open(e1) as ds: X1=ds.read().astype("float32"); nd1=ds.nodata
    valid &= np.all(np.isfinite(X0),axis=0)&np.all(np.isfinite(X1),axis=0)
    if nd0 is not None: valid &= np.all(X0!=nd0,axis=0)
    if nd1 is not None: valid &= np.all(X1!=nd1,axis=0)
    rr,cc=np.where(valid); xs,ys=rasterio.transform.xy(tr,rr,cc,offset="center")
    d=pd.DataFrame({"site":site,"landscape":row.landscape,"obs_year":year,"support_m":support,
                    "row":rr,"col":cc,"x":np.asarray(xs),"y":np.asarray(ys),
                    "uav_valid_fraction":cov[valid],"musanga_fraction":y[valid]})
    for j in range(64):
        d[f"T0_A{j:02d}"]=X0[j][valid]; d[f"T1_A{j:02d}"]=X1[j][valid]
    return d

tables=[]
for row in inventory.itertuples(index=False):
    cache=OUT/"tables"/f"{row.site}_10m.csv"; cache.parent.mkdir(parents=True,exist_ok=True)
    if cache.exists() and not FORCE:
        print("Reusing",cache.name); d=pd.read_csv(cache)
    else:
        print("Building",row.site); d=build_site_table(row,10); d.to_csv(cache,index=False)
    tables.append(d)
df10=pd.concat(tables,ignore_index=True)
print("10 m rows:",len(df10))
display(df10.groupby(["landscape","site"]).musanga_fraction.agg(["count","mean","median","max"]))
df10.to_csv(OUT/"multisite_10m_T0_T1.csv",index=False)

## BLOCK 06 — Metrics: continuous fractional cover + detection diagnostics

In [ ]:
COVER_THRESHOLDS=[0.01,0.05,0.20,0.50]

def reg_metrics(y,p):
    rho=spearmanr(y,p).statistic if len(y)>=3 else np.nan
    return {"RMSE":mean_squared_error(y,p)**0.5,"MAE":mean_absolute_error(y,p),
            "R2":r2_score(y,p),"Spearman":rho,"bias":float(np.mean(p-y))}

def best_f1_cut(y,p,truth_thr):
    z=(y>truth_thr).astype(int)
    best=(0.5,-1)
    for cut in np.linspace(0,1,201):
        f=f1_score(z,p>cut,zero_division=0)
        if f>best[1]: best=(cut,f)
    return best[0]

def detection_metrics(y,p,truth_thr,pred_cut):
    z=(y>truth_thr).astype(int)
    if len(np.unique(z))<2:
        return {"ROC_AUC":np.nan,"PR_AUC":np.nan,"F1":np.nan,"precision":np.nan,
                "recall":np.nan,"balanced_accuracy":np.nan,"pred_cut":pred_cut}
    zh=(p>pred_cut).astype(int)
    return {"ROC_AUC":roc_auc_score(z,p),"PR_AUC":average_precision_score(z,p),
            "F1":f1_score(z,zh,zero_division=0),"precision":precision_score(z,zh,zero_division=0),
            "recall":recall_score(z,zh,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(z,zh),"pred_cut":pred_cut}

print("Metrics ready.")

## BLOCK 07 — E01/E02/E03: strict leave-one-landscape-out at 10 m

For the Mos products, all three mosaics are held out together as one landscape. This avoids treating adjacent mosaics as independent sites.

In [ ]:
FEATURE_SETS={
    "T0":[f"T0_A{i:02d}" for i in range(64)],
    "T+1":[f"T1_A{i:02d}" for i in range(64)],
    "T0+T1":[f"T0_A{i:02d}" for i in range(64)]+[f"T1_A{i:02d}" for i in range(64)]
}
landscapes=sorted(df10.landscape.unique())
reg_rows=[]; det_rows=[]
for fs,cols in FEATURE_SETS.items():
    for hold in landscapes:
        tr=df10.landscape!=hold; te=df10.landscape==hold
        model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",
                                    n_jobs=-1,random_state=RANDOM_SEED)
        model.fit(df10.loc[tr,cols].values,df10.loc[tr,"musanga_fraction"].values)
        ptr=np.clip(model.predict(df10.loc[tr,cols].values),0,1)
        pte=np.clip(model.predict(df10.loc[te,cols].values),0,1)
        ytr=df10.loc[tr,"musanga_fraction"].values; yte=df10.loc[te,"musanga_fraction"].values
        reg_rows.append({"features":fs,"held_out":hold,"n_test":len(yte),**reg_metrics(yte,pte)})
        for thr in COVER_THRESHOLDS:
            cut=best_f1_cut(ytr,ptr,thr)  # training only
            det_rows.append({"features":fs,"held_out":hold,"truth_cover_threshold":thr,
                             "n_test":len(yte),**detection_metrics(yte,pte,thr,cut)})
reg10=pd.DataFrame(reg_rows); det10=pd.DataFrame(det_rows)
print("Continuous metrics"); display(reg10)
print("Detection diagnostics"); display(det10)
reg10.to_csv(OUT/"results_10m_regression.csv",index=False)
det10.to_csv(OUT/"results_10m_detection.csv",index=False)

## BLOCK 08 — Resolution experiment scaffold: 10 / 20 / 30 / 50 m

Run this only after the 10 m baseline above looks sensible. Each support is rebuilt directly from native UAV masks and annual embeddings; it is not derived by averaging the 10 m target.

In [ ]:
resolution_tables={}
for support in SUPPORTS:
    if support==10:
        resolution_tables[10]=df10
        continue
    pieces=[]
    for row in inventory.itertuples(index=False):
        cache=OUT/"tables"/f"{row.site}_{support}m.csv"; cache.parent.mkdir(parents=True,exist_ok=True)
        if cache.exists() and not FORCE:
            d=pd.read_csv(cache)
        else:
            print("Building",row.site,support,"m")
            d=build_site_table(row,support); d.to_csv(cache,index=False)
        pieces.append(d)
    resolution_tables[support]=pd.concat(pieces,ignore_index=True)
    print(support,"m rows:",len(resolution_tables[support]))
print("Resolution target tables ready.")

## BLOCK 09 — Resolution benchmark using the best temporal configuration

Set `BEST_TEMPORAL` after inspecting BLOCK 07. The default is T0, the observation-year embedding.

In [ ]:
BEST_TEMPORAL="T0"
cols=FEATURE_SETS[BEST_TEMPORAL]
support_rows=[]
for support,d in resolution_tables.items():
    for hold in sorted(d.landscape.unique()):
        tr=d.landscape!=hold; te=d.landscape==hold
        model=RandomForestRegressor(n_estimators=400,min_samples_leaf=2,max_features="sqrt",
                                    n_jobs=-1,random_state=RANDOM_SEED)
        model.fit(d.loc[tr,cols].values,d.loc[tr,"musanga_fraction"].values)
        p=np.clip(model.predict(d.loc[te,cols].values),0,1)
        y=d.loc[te,"musanga_fraction"].values
        support_rows.append({"support_m":support,"features":BEST_TEMPORAL,
                             "held_out":hold,"n_test":len(y),**reg_metrics(y,p)})
support_results=pd.DataFrame(support_rows)
display(support_results)
support_results.to_csv(OUT/"results_resolution_regression.csv",index=False)

## BLOCK 10 — Next stage

Do **not** add Sentinel-2 until the embedding-only baseline has been inspected.

Decision sequence:
1. Is fractional cover predictable at 10 m across held-out landscapes?
2. Is T0, T+1, or T0+T1 most robust?
3. Does 20/30/50 m support materially improve transfer?
4. Then add Sentinel-2-only and Embedding + Sentinel-2 benchmarks using the same targets and validation folds.